# Build the joint CTCL/MF/SS multi-tissue atlas + TCR clone table

Joins the **Li 2024 skin atlas** (`CTCL_all`, 419k cells) with the standalone GEO cohorts
(`D1 D3 H D5 D6 B1 B2 B4`) into one full-compartment atlas, with a harmonized `obs` schema and
a cross-platform **TCR clone table** (CDR3-based, TRB-primary).

**Excluded:** `B3` (453-gene targeted panel + no CDR3, only a paired-chain boolean), `D2` (no data yet).

Schema + crosswalk: [`data/atlas_joint/metadata_schema.md`](data/atlas_joint/metadata_schema.md).
All logic lives in `atlas_join_helpers.py`.

**Run order.** Steps 1-4 are GPU-kernel jobs on `neural_nmf_env`; heavy outputs are cached to disk
(re-runs skip). Set the `FORCE_*` flags to rebuild.

| Step | Output |
|---|---|
| 1 | `data/<LABEL>/processed/standardized.h5ad` |
| 2 | `data/atlas_joint/tcr_clones.parquet` |
| 3 | `data/atlas_joint/joint_raw.h5ad` |
| 4 | `data/atlas_joint/joint_integrated.h5ad` |
| 5 | `figures/joint_umap_*.png` + `data/atlas_joint/README.md` |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad


def _resolve_nb_dir():
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(start)


NB_DIR = _resolve_nb_dir()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
import importlib
import atlas_join_helpers as H
importlib.reload(H)   # pick up edits without restarting the kernel

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1
OUT = NB_DIR / "data" / "atlas_joint"
OUT.mkdir(exist_ok=True)
REG = H.dataset_registry(NB_DIR)
EXPR = [k for k, v in REG.items() if v.get("in_expression")]
print("NB_DIR:", NB_DIR)
print("expression datasets:", EXPR)

## Step 1 - per-dataset standardized h5ads

Each source -> one AnnData with exactly `CANON_OBS` columns, `X` = raw counts, gene-symbol
`var_names`, and a `cell_id` consistent with the TCR table. Reuses the existing `concat.h5ad`
for D1/D3; rebuilds H (ECCITE, all 14 samples) and D5/D6/B1/B2/B4 from `raw/`; maps the atlas.

In [ ]:
# Per-paper QC (QC_CONFIG) is applied inside build_standardized: per-cell
# min/max-genes + %mito cutoffs, min_cells_per_gene=3, per-sample Scrublet.
# li24 is prefiltered (atlas already QC'd) -> metrics only.
#
# CACHING: each standardized.h5ad is written as soon as its dataset finishes, and
# FORCE_STD=False reuses it -> a rerun resumes after a crash (only redoes missing
# datasets), instead of rebuilding all 9 (~45 min). Set FORCE_STD=True only to
# force a full rebuild. The loop is crash-resilient: a failing dataset is recorded
# and skipped so the rest still complete.
FORCE_STD = False
RUN_SCRUBLET = True
std_paths, failed = {}, {}
for label in EXPR:
    try:
        std_paths[label] = H.build_standardized(label, NB_DIR, force=FORCE_STD,
                                                run_scrublet=RUN_SCRUBLET)
    except Exception as e:  # keep the already-built datasets; report the failure
        failed[label] = repr(e)
        print(f"  [{label}] FAILED: {e!r}")
if failed:
    print("\nFAILED datasets (rerun this cell after fixing):", list(failed))
print("built:", list(std_paths))
std_paths

### QC audit

Per-dataset cells in → kept (from `uns['qc']`), with the threshold source. Lets you confirm the
filtering matches each paper and see how much `geskin26`/`gaydosik19` shrink toward their reported
counts. (For Scrublet auto-thresholding, `%pip install scikit-image`; otherwise a fixed 0.25 cutoff
is used.)

In [ ]:
rows = []
for label, p in std_paths.items():
    u = sc.read_h5ad(p, backed="r").uns.get("qc", {})
    n_in, n_kept = u.get("n_in", 0), u.get("n_kept", 0)
    rows.append(dict(dataset=H.DATASET_NAME.get(label, label), n_in=n_in, n_kept=n_kept,
                     pct_kept=round(100 * n_kept / max(1, n_in), 1),
                     doublets=u.get("n_doublet", 0), source=u.get("source", "")))
qc_audit = pd.DataFrame(rows).set_index("dataset")
print(qc_audit.to_string())
qc_audit

## Step 2 - unified TCR clone table

CDR3 exact key, **TRB-primary** (TRA dropout does not split a clone). 10x VDJ: D1/D3/D5/B2/B4;
ECCITE membership matrix: H. Per-sample dominant clone (>=5% of VDJ cells **and** >=2x the 2nd
clone) -> `is_malignant` (generalizes `07_d1`). B1/D6 have no TCR; B3 has no CDR3.

In [ ]:
FORCE_TCR = False
tcr_parquet = H.build_tcr_table(NB_DIR, force=FORCE_TCR)
tcr = pd.read_parquet(tcr_parquet)
tcr.groupby("dataset", observed=True).agg(
    cells=("cell_id", "size"), clones=("clone_id", "nunique"),
    malignant=("is_malignant", "sum"))

### (optional) scirpy fuzzy clonotype merging

The exact TRB-CDR3 key above is the production path. scirpy can additionally merge 1-aa CDR3
variants into clonotype clusters. Opt-in (needs `%pip install scirpy`); validate against your
installed scirpy version before trusting the refined ids. Leave OFF to use the exact key.

In [ ]:
USE_SCIRPY = False
if USE_SCIRPY:
    try:
        import scirpy as ir
        print("scirpy", ir.__version__, "- refine clone_id per (dataset, donor) with ir_dist/define_clonotype_clusters")
        # NOTE: scirpy API varies across versions; implement + validate here before use.
    except ImportError:
        print("scirpy not installed; run %pip install scirpy")
else:
    print("Using exact TRB-CDR3 clone key (scirpy refinement disabled).")

## Step 3 - dedup overlap + concatenate

`detect_duplicate_cells` reports the shared Brunner patients (D1==D3 reuse the same libraries;
`P112` is byte-identical). `concat_joint` keeps **D1's** copy of `P112/P115/P116/P121/P303/PGS`,
drops the D3 duplicates, outer-joins all datasets on gene symbols, and left-joins the TCR table.

In [ ]:
print("shared-Brunner cell counts per dataset:")
print(H.detect_duplicate_cells(std_paths))

FORCE_CONCAT = False
joint_path = H.concat_joint(NB_DIR, std_paths, tcr_parquet,
                            force=FORCE_CONCAT, brunner_keep="D1")

# cache the concatenated atlas in-session (joint_raw.h5ad is already on disk)
adata_raw = sc.read_h5ad(joint_path)
# friendly dataset names (author+year); relabels an older cache in place, no rebuild
_friendly = adata_raw.obs["dataset"].map(lambda d: H.DATASET_NAME.get(d, d)).astype("category")
if not adata_raw.obs["dataset"].astype(str).equals(_friendly.astype(str)):
    adata_raw.obs["dataset"] = _friendly
    adata_raw.write_h5ad(joint_path)
    print("relabeled dataset names ->", list(adata_raw.obs["dataset"].cat.categories))
print("joint_raw:", adata_raw.shape,
      "| TCR+ cells:", int(adata_raw.obs["has_tcr"].sum()),
      "| malignant:", int(adata_raw.obs["is_malignant"].sum()))
adata_raw

## Step 3b·a — apply final curated metadata (`sample_metadata_final.csv`) → cache

Rebuilds `joint_annotated.h5ad` from the pristine concat (`joint_raw.h5ad`) using the
**authoritative** `data/sample_metadata_final.csv`: `H.annotate_from_csv` broadcasts the curated
per-sample fields (`repo, accession, stage_class, lineage, entity, cohort_country/region,
origin_resolve, east_asian, …`) onto `obs` by `sample_id`, then `H.resolve_lineage` finalizes the
provisional lineages (`lineage_resolve != "no"`), per a priority order:

1. **D1 clinical** (`patients.tsv`) — authoritative. The Chennareddy `subset` diagnosis
   (Berti→CD8, gd_MF→γδ, CD4_MF→CD4) overrides expression, which mis-calls the γδ cases
   (γδ malignant cells under-express TRGC/TRDC; tiny malignant fractions give noisy gd ratios).
   Expression scores are still recorded and `lineage_disagree` flags where they differ.
2. **Literature-resolved** (`resolved_*`, e.g. Liu2022, incl. MF14=CD8) — kept as-is.
3. **Expression / inferCNV** (the TCR-less li24 `PT*`) — γδ if gd/(gd+ab)≥thresh else CD8 if
   CD8>CD4 else CD4, marked `needs_infercnv_verification` (malignant label not TCR-backed).
4. **Unresolvable** (no malignant cells) — left unchanged, `lineage_resolve="unresolved"`.

`entity` is synced to the resolved lineage (curated non-placeholder entities like Sezary /
MF/SS_leukemic preserved); `lineage_resolve` is rewritten to `resolved`/`unresolved`.

Starting from `joint_raw` (not the prior annotated cache) keeps the **per-cell** base columns —
notably per-cell `tissue` (Epidermis/Dermis) — intact. The cell validates sample coverage +
disease/compartment vocab + non-null lineage, asserts no stale `*_RESOLVE` placeholders and that
`entity`/`lineage_resolve`/`lineage_method` are mutually consistent, then re-writes the cache.

In [ ]:
# Merge the authoritative curated metadata (sample_metadata_final.csv) onto the atlas.
# Canonical path: start from joint_raw (pristine per-cell base obs, incl. per-cell tissue),
# broadcast the curated per-sample fields with annotate_from_csv, then resolve provisional lineages.
# D1 lineages come from the Chennareddy clinical table (patients.tsv) — authoritative — not from
# expression; entity is synced to the resolved lineage; lineage_resolve -> resolved/unresolved.
ANNOT = OUT / "joint_annotated.h5ad"
FINAL = NB_DIR / "data" / "sample_metadata_final.csv"
D1_CLIN = NB_DIR / "data" / "D1_chennareddy2025" / "meta" / "patients.tsv"

adata_raw = sc.read_h5ad(OUT / "joint_raw.h5ad")            # per-cell base columns intact
adata_raw, added = H.annotate_from_csv(adata_raw, FINAL)    # adds curated cols (repo ... east_asian)
report = H.resolve_lineage(adata_raw, min_cells=20, gd_frac_thresh=0.5, clinical_d1_tsv=D1_CLIN)
print("\nlineage resolution (lineage_resolve != 'no'):")
print(report.to_string(index=False))
flips = report[report["disagree"]]
if len(flips):
    print(f"\n{len(flips)} sample(s) where the expression call disagrees with the clinical diagnosis "
          "(clinical wins; expr scores retained for audit):")
    print(flips[["sample_id", "expr_call", "clinical", "final"]].to_string(index=False))

# make sure everything is in order
meta_ids = set(pd.read_csv(FINAL, dtype=str)["sample_id"])
missing = sorted(set(adata_raw.obs["sample_id"].astype(str).unique()) - meta_ids)
assert not missing, f"{len(missing)} obs samples absent from CSV: {missing[:10]}"
assert not (set(adata_raw.obs["disease"].unique()) - H.DISEASE_VOCAB), "bad disease vocab"
assert not (set(adata_raw.obs["compartment"].unique()) - H.COMPARTMENT_VOCAB), "bad compartment vocab"
assert adata_raw.obs["lineage"].isna().sum() == 0, "NaN lineage after resolve"
for c in ["cohort_country", "cohort_region", "origin_resolve", "east_asian"]:
    assert c in adata_raw.obs.columns, f"missing new column: {c}"

# post-fix validation: no stale *_RESOLVE placeholders; lineage_resolve↔lineage_method agree;
# entity consistent with lineage on every resolved row.
o = adata_raw.obs
for col in ("lineage", "entity", "lineage_resolve"):
    bad = int(o[col].astype(str).str.contains("RESOLVE").sum())
    assert bad == 0, f"{bad} cells still carry a *_RESOLVE placeholder in {col}"
assert not o["lineage_resolve"].astype(str).str.startswith("yes").any(), "stale 'yes_*' lineage_resolve"
res, meth = o["lineage_resolve"].astype(str), o["lineage_method"].astype(str)
assert meth[res == "unresolved"].str.startswith("unresolved").all(), "unresolved rows: method mismatch"
assert meth[res == "resolved"].str.startswith("resolved").all(), "resolved rows: method mismatch"
assert (meth[res == "no"] == "high_confidence").all(), "'no' rows: method mismatch"
ent_ok = {"CD8": {"MF_CD8", "CD8_aggressive_epidermotropic_CTCL_Berti"},
          "gamma_delta": {"MF_gamma_delta"}}
sub = o[(res == "resolved") & o["lineage"].astype(str).isin(ent_ok)]
for lin, allowed in ent_ok.items():
    got = set(sub.loc[sub["lineage"].astype(str) == lin, "entity"].astype(str).unique())
    assert got <= allowed, f"lineage {lin} maps to inconsistent entity {got - allowed}"
print("\nlineage × entity (non-empty rows):")
print(pd.crosstab(o["lineage"], o["entity"]).loc[lambda d: d.sum(1) > 0].to_string())

adata_raw.write_h5ad(ANNOT)
print(f"\n[merge] added {len(added)} curated cols: {added}")
print(f"[merge] wrote {ANNOT}  shape={adata_raw.shape}  obs cols={adata_raw.obs.shape[1]}")

In [ ]:
# Refine the cached lineage: D1 clinical override + hold the TCR-less PT* as provisional, then re-cache.
# Runs after the H.resolve_lineage cell (operates on the in-memory adata_raw) and writes back to
# joint_annotated.h5ad so the export cell below (which reloads from disk) sees it.
import pandas as pd

obs = adata_raw.obs

# columns we edit -> plain strings so re-assignment doesn't hit categorical-membership errors
for c in ['lineage', 'entity', 'lineage_method', 'lineage_resolve']:
    obs[c] = obs[c].astype(str)

# per-cell patient key = the part of sample_id after the dataset prefix. For D1 this is
# `sample_id_in_raw` (see CANON build), so match patients.tsv on that column, no suffix stripping.
pat = obs['sample_id'].astype(str).str.split('__').str[-1]

# ---- 1. D1 (Chennareddy): clinical diagnosis overrides the expression call ----
clin   = pd.read_csv(NB_DIR / 'data' / 'D1_chennareddy2025' / 'meta' / 'patients.tsv', sep='\t')
PATCOL = 'sample_id_in_raw'                       # matches the atlas sample_id suffix
sub2   = {'gd_MF': ('gamma_delta', 'MF_gamma_delta'),
          'Berti': ('CD8', 'CD8_aggressive_epidermotropic_CTCL')}
clin_map = {p: sub2[s] for p, s in zip(clin[PATCOL], clin['subset']) if s in sub2}

expr  = obs['lineage'].copy()                    # expression call, before override
is_d1 = obs['dataset'].eq('chennareddy25') & pat.isin(set(clin_map))
for p, (lin, ent) in clin_map.items():
    m = is_d1 & pat.eq(p)
    obs.loc[m, 'lineage'] = lin
    obs.loc[m, 'entity']  = ent
obs['lineage_disagreement'] = is_d1 & obs['lineage'].ne(expr) & ~expr.str.contains('RESOLVE')
obs.loc[is_d1, 'lineage_method']  = 'clinical(patients.tsv)'
obs.loc[is_d1, 'lineage_resolve'] = 'resolved_clinical'

# ---- 2. PT* atlas (no TCR -> inferCNV-based): hold provisional, don't lock CD8/gd ----
pt = obs['dataset'].eq('li24') & pat.isin(['PT11', 'PT53', 'PT55'])
obs.loc[pt, 'lineage']         = 'unresolved_provisional'   # scores retained in lineage_score_*
obs.loc[pt, 'entity']          = 'MF_unresolved'
obs.loc[pt, 'lineage_method']  = 'inferCNV_state_no_TCR(unverified)'
obs.loc[pt, 'lineage_resolve'] = 'provisional_inferCNV_unverified'

# ---- 3. drop the broken collapse columns only if they leaked into obs ----
for c in ['is_malignant', 'is_dominant_clone']:
    if c in obs and obs[c].astype(str).eq('2 values').any():
        obs.drop(columns=c, inplace=True)

# restore categoricals + persist so the export cell (reloads from disk) picks up the refinement
for c in ['lineage', 'entity', 'lineage_method', 'lineage_resolve']:
    obs[c] = obs[c].astype('category')
adata_raw.write_h5ad(OUT / "joint_annotated.h5ad")

# sanity check
print(obs.loc[is_d1 | pt, ['lineage', 'entity', 'lineage_resolve']].value_counts())

In [ ]:
# Step 3b - read & print the final cached annotated atlas + re-export per-sample metadata (for comparison)
import re

adata_raw = sc.read_h5ad(OUT / "joint_annotated.h5ad")
print(adata_raw)

obs = adata_raw.obs
# sanity: per-cell booleans (is_malignant, is_dominant_clone, ...) must stay per-cell in obs,
# never the literal "N values" / "N unique" collapse string.
_collapsed = re.compile(r"^\d+ (values|unique)$")
leaked = [c for c in obs.columns
          if obs[c].dtype == object and obs[c].astype(str).str.match(_collapsed).any()]
assert not leaked, f"obs columns hold a collapsed-aggregate string: {leaked}"

def _agg_col(s):            # per-sample summary that never yields a bare "N values"
    if pd.api.types.is_bool_dtype(s):
        return round(float(s.mean()), 4)               # per-cell boolean -> fraction True
    if s.name == "tissue":                             # legitimately multi-valued -> canonical join
        return "|".join(sorted({H._norm_tissue(x) for x in s.dropna()}))
    u = pd.unique(s.dropna())
    if len(u) == 1:
        return u[0]                                     # genuine per-sample scalar
    if pd.api.types.is_numeric_dtype(s):
        return round(float(s.mean()), 4)               # varying numeric (scores) -> mean
    return f"{len(u)} unique"                           # varying per-cell categorical (cell_id, cdr3)

# aggregate *every* obs column (complete quality check), then append derived counts/fractions.
g = obs.groupby("sample_id", observed=True)
sample_meta = g.agg(**{c: (c, _agg_col) for c in obs.columns if c != "sample_id"})
for name, col, fn in [("n_donors", "donor", lambda s: s.nunique()),
                      ("n_cells", "cell_id", lambda s: s.size),
                      ("n_tcr", "has_tcr", lambda s: int(s.sum())),
                      ("n_clones", "clone_id", lambda s: s[s != ""].nunique()),
                      ("n_malignant", "is_malignant", lambda s: int(s.sum()))]:
    if col in obs.columns:
        sample_meta[name] = g[col].agg(fn)
sample_meta = sample_meta.reset_index()
sort_cols = [c for c in ["dataset", "sample_id"] if c in sample_meta.columns]
sample_meta = sample_meta.sort_values(sort_cols)

# (a) no "N values" anywhere; (b) entity consistent with lineage; (c) lineage_resolve↔lineage_method
assert not sample_meta.astype(str).apply(lambda c: c.str.match(r"^\d+ values$")).to_numpy().any(), \
    "a column still collapsed to a bare 'N values' string"
ent_ok = {"CD8": {"MF_CD8", "CD8_aggressive_epidermotropic_CTCL_Berti"},
          "gamma_delta": {"MF_gamma_delta"}}
for lin, allowed in ent_ok.items():
    got = set(sample_meta.loc[sample_meta["lineage"].astype(str) == lin, "entity"].astype(str))
    assert got <= allowed, f"per-sample lineage {lin} maps to inconsistent entity {got - allowed}"
agree = sample_meta.apply(
    lambda r: (str(r["lineage_method"]).startswith(str(r["lineage_resolve"]))
               if r["lineage_resolve"] in ("resolved", "unresolved")
               else str(r["lineage_method"]) == "high_confidence"), axis=1)
assert agree.all(), f"lineage_resolve/lineage_method disagree:\n{sample_meta.loc[~agree, ['sample_id','lineage_resolve','lineage_method']]}"

# write to a *separate* file so the authoritative input (sample_metadata.csv) is untouched
csv_path = OUT / "sample_metadata_check.csv"
sample_meta.to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(sample_meta)} samples, {sample_meta.shape[1]} cols)  -- checks: no 'N values', entity↔lineage, resolve↔method all pass")
sample_meta

## Step 4 — Raw UMAP (baseline, before MrVI)

Loads the cached annotated atlas and computes a **standard UMAP straight from raw counts**
(HVG → normalize → log1p → PCA → neighbors → UMAP) over all ~1.34M cells — no integration. This is
the un-corrected baseline (expect visible `study`/`compartment` batch structure) to compare against
the MrVI `X_mrvi_u` embedding in Step 6.

**HEAVY:** run on the GPU kernel. UMAP coordinates are cached to
`data/atlas_joint/joint_umap_raw.npy` so re-runs just reload.

In [ ]:
# Step 4 (pre-UMAP) - atlas overview: headline counts + obs summary tables
ANNOT = OUT / "joint_annotated.h5ad"
if "adata_raw" not in globals():
    adata_raw = sc.read_h5ad(ANNOT)
obs = adata_raw.obs


def _col(name):
    return obs[name] if name in obs.columns else None


print("=" * 60)
print("JOINT CTCL/MF/SS ATLAS - overview")
print("=" * 60)
print(f"cells        : {adata_raw.n_obs:,}")
print(f"genes        : {adata_raw.n_vars:,}")
for label, c in [("studies", "study"), ("datasets", "dataset"), ("samples", "sample_id"),
                 ("donors", "donor"), ("real donors", "real_donor"),
                 ("cell types", "cell_type"), ("entities", "entity")]:
    s = _col(c)
    if s is not None:
        print(f"{label:<12} : {s.nunique():,}")

# TCR / malignancy headline
if "has_tcr" in obs:
    n_tcr = int(obs["has_tcr"].sum())
    print(f"TCR+ cells   : {n_tcr:,}  ({100 * n_tcr / adata_raw.n_obs:.1f}%)")
if "is_malignant" in obs and not obs["is_malignant"].astype(str).eq("2 values").any():
    n_mal = int(obs["is_malignant"].sum())
    print(f"malignant    : {n_mal:,}  ({100 * n_mal / adata_raw.n_obs:.1f}%)")
if "clone_id" in obs:
    print(f"TCR clones   : {obs.loc[obs['clone_id'].astype(str) != '', 'clone_id'].nunique():,}")


def summary_table(col, top=None):
    """Per-category cell counts + % + #samples, sorted by cell count."""
    if col not in obs.columns:
        return None
    g = obs.groupby(col, observed=True)
    df = pd.DataFrame({
        "cells": g.size(),
        "pct": (100 * g.size() / adata_raw.n_obs).round(1),
    })
    if "sample_id" in obs.columns:
        df["samples"] = g["sample_id"].nunique()
    if "study" in obs.columns:
        df["studies"] = g["study"].nunique()
    df = df.sort_values("cells", ascending=False)
    return df.head(top) if top else df


for col in ["study", "disease", "stage_class", "compartment", "tissue",
            "lineage", "sex", "tech"]:
    df = summary_table(col)
    if df is not None:
        print(f"\n--- by {col} ---")
        print(df.to_string())

print("\n--- top 15 cell types ---")
ct = summary_table("cell_type", top=15)
if ct is not None:
    print(ct.to_string())

# disease x compartment cross-tab (where the biology vs batch confound lives)
if {"disease", "compartment"} <= set(obs.columns):
    print("\n--- disease x compartment (cells) ---")
    print(pd.crosstab(obs["disease"], obs["compartment"]).to_string())


In [ ]:
# Step 4 - load cached annotated atlas + baseline UMAP from raw counts (no integration)
import numpy as np

ANNOT = OUT / "joint_annotated.h5ad"
if "adata_raw" not in globals():
    adata_raw = sc.read_h5ad(ANNOT)
print(adata_raw)

np.random.seed(SEED)
UMAP_NPY = OUT / "joint_umap_raw.npy"
if UMAP_NPY.exists():
    adata_raw.obsm["X_umap"] = np.load(UMAP_NPY)
    print("loaded cached UMAP", adata_raw.obsm["X_umap"].shape)
else:
    # HEAVY (GPU kernel): standard pipeline on a 2k-HVG copy (keeps adata_raw genes intact)
    hvg = sc.pp.highly_variable_genes(
        adata_raw, n_top_genes=2000, flavor="seurat_v3",
        layer="raw_counts", inplace=False, subset=False,
    )
    ad_vis = adata_raw[:, hvg["highly_variable"].to_numpy()].copy()
    ad_vis.X = ad_vis.layers["raw_counts"].copy()
    sc.pp.normalize_total(ad_vis, target_sum=1e4)
    sc.pp.log1p(ad_vis)
    sc.pp.scale(ad_vis, max_value=10)
    sc.pp.pca(ad_vis, n_comps=50, random_state=SEED)
    sc.pp.neighbors(ad_vis, random_state=SEED)
    sc.tl.umap(ad_vis, random_state=SEED)
    adata_raw.obsm["X_umap"] = ad_vis.obsm["X_umap"]
    np.save(UMAP_NPY, adata_raw.obsm["X_umap"])
    print("computed + cached UMAP ->", UMAP_NPY)

In [ ]:
# Step 4b - baseline UMAP colored by annotation (before MrVI)
sc.pl.umap(
    adata_raw,
    color=["cell_type", "study", "disease", "compartment"],
    ncols=2, frameon=False, size=2, wspace=0.25,
)

## Step 6 — MrVI sample-level model

Fits **MrVI** (`scvi.external.MRVI`) on the full joint atlas → a sample-aware latent space plus
sample-level comparative analyses (sample distances, differential abundance, DE).

- `sample_key="sample_id"` (109 samples), `batch_key="study"` (9 cohorts), `layer="raw_counts"`.
  Per the MrVI paper, the sample/donor identifier is the natural target covariate and the
  *study of origin* is the textbook cross-study nuisance.
- Input HVG-subset to 10k genes (`seurat_v3`, batched on `study`), cached to
  `data/atlas_joint/joint_mrvi_input.h5ad`. Architecture/training use the MrVI defaults
  (`n_latent=30`, `n_latent_u=10`, `max_epochs=100` w/ early stopping, `batch_size=256`,
  `lr=2e-3`) — set "through hyperparameter sensitivity studies" in the paper.

**Caveat — study/disease confound.** This MRVI build accepts a *single* `batch_key`. In this atlas
`study` is partly confounded with `disease`/`compartment` (blood-only Sézary vs skin-only MF cohorts),
so the `study` nuisance can absorb biological signal. Interpret comparative DE/DA across `disease`
**within strata** where both groups co-occur (e.g. MF-skin stage contrasts); the confound-free
contrasts are **within-donor** (malignant-vs-benign pseudo-samples, see `jobs/run_mrvi_tcells_mb.py`).

### Launch (GPU, bsub)

Training on 1.34M cells is heavy — run as a job, **not** in this kernel:

```bash
bash jobs/run_mrvi_joint.sh                       # full atlas, 15k HVG, max_epochs=50
bash jobs/run_mrvi_joint.sh --subsample 300000    # faster: fit on 300k cells, transform all
bash jobs/run_mrvi_joint.sh --skip-train          # reuse models/mrvi_joint, just (re)cache latents/diagnostics
```

Monitor with `tail -f jobs/run_mrvi_joint.bsub.log`. Outputs: `models/mrvi_joint/`,
`data/atlas_joint/joint_X_mrvi{,_u}.npy`, `figures/mrvi_joint_{history.json,donor_distances.nc,da.nc}`.
The HVG-subset input is cached once to `data/atlas_joint/joint_mrvi_input.h5ad` (rebuild with `--force`).

In [ ]:
# Step 6a - submit the MrVI training job to the GPU queue (bsub returns immediately)
import subprocess

res = subprocess.run(
    ["bash", str(NB_DIR / "jobs" / "run_mrvi_joint.sh")],
    capture_output=True, text=True,
)
print(res.stdout, res.stderr)   # prints the submitted job id
# monitor:  !tail -n 40 jobs/run_mrvi_joint.bsub.log
# faster first pass:  add ["--subsample", "300000"] to the command above

In [ ]:
# Step 6b - load cached MrVI latents into obsm (after the bsub job finishes)
import numpy as np

adata_raw.obsm["X_mrvi_z"] = np.load(OUT / "joint_X_mrvi.npy")     # sample-aware
adata_raw.obsm["X_mrvi_u"] = np.load(OUT / "joint_X_mrvi_u.npy")   # sample-unaware
print("X_mrvi_z", adata_raw.obsm["X_mrvi_z"].shape,
      "| X_mrvi_u", adata_raw.obsm["X_mrvi_u"].shape)

# HEAVY - run on the GPU kernel only: neighbors+UMAP on 1.34M cells.
# X_mrvi_u = integration view (cell state); X_mrvi_z = sample-aware view.
sc.pp.neighbors(adata_raw, use_rep="X_mrvi_u")
sc.tl.umap(adata_raw)
sc.pl.umap(adata_raw, color=["cell_type", "study", "disease", "compartment"], ncols=2)

In [ ]:
# Step 6c - sample-distance + differential-abundance diagnostics (after the job finishes)
import xarray as xr
import matplotlib.pyplot as plt

FIG = NB_DIR / "figures"
dist = xr.open_dataset(FIG / "mrvi_joint_donor_distances.nc")
da = xr.open_dataset(FIG / "mrvi_joint_da.nc")
print(dist)
print(da)

# mean sample x sample distance (averaged over cell-type groups) - does it cluster by study?
dvar = next(iter(dist.data_vars))
arr = dist[dvar]
M = arr.mean(dim=arr.dims[0]).to_pandas()           # collapse the cell_type group dim
fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(M.values, cmap="viridis")
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("MrVI sample distances: check disease vs study clustering")
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()

## Step 7 — MrVI (skin-only)

Re-runs the **same** pipeline on skin cells (`compartment == "Skin"`: 79 samples, ~866k cells,
6 studies) — smaller and less study/disease-confounded than the full atlas (drops the blood-only
Sézary cohorts + the single LN sample). Reuses `jobs/run_mrvi_joint.py` via
`--subset-compartment Skin --tag skin`; every output gets a `_skin` suffix
(`joint_mrvi_input_skin.h5ad`, `models/mrvi_joint_skin/`, `joint_X_mrvi{,_u}_skin.npy`,
`mrvi_joint_skin_*`) so it never collides with the full-atlas run. The submit cell sets
`JOB_TAG=skin` so the bsub job name + log are also distinct (`run_mrvi_joint_skin.bsub.log`).

In [ ]:
# Step 7a - submit the skin-only MrVI job (JOB_TAG gives it a distinct job name + log)
import os, subprocess

env = {**os.environ, "JOB_TAG": "skin", "WALL_H": "48"}
res = subprocess.run(
    ["bash", str(NB_DIR / "jobs" / "run_mrvi_joint.sh"),
     "--subset-compartment", "Skin", "--tag", "skin"],
    capture_output=True, text=True, env=env,
)
print(res.stdout, res.stderr)   # submitted job id
# monitor:  !tail -n 40 jobs/run_mrvi_joint_skin.bsub.log

In [ ]:
# Step 7b - load skin MrVI latents (after the skin job finishes)
import numpy as np

ad_skin = adata_raw[adata_raw.obs["compartment"] == "Skin"].copy()   # same row order as the job
ad_skin.obsm["X_mrvi_z"] = np.load(OUT / "joint_X_mrvi_skin.npy")     # sample-aware
ad_skin.obsm["X_mrvi_u"] = np.load(OUT / "joint_X_mrvi_u_skin.npy")   # sample-unaware
print("ad_skin", ad_skin.shape,
      "| X_mrvi_z", ad_skin.obsm["X_mrvi_z"].shape,
      "| X_mrvi_u", ad_skin.obsm["X_mrvi_u"].shape)

# HEAVY - GPU kernel: neighbors+UMAP on the skin subset (X_mrvi_u = integration view)
# sc.pp.neighbors(ad_skin, use_rep="X_mrvi_u")
# sc.tl.umap(ad_skin)
# sc.pl.umap(ad_skin, color=["cell_type", "study", "disease", "stage_class"], ncols=2)

In [ ]:
sc.pp.neighbors(ad_skin, use_rep="X_mrvi_u")
sc.tl.umap(ad_skin)
sc.pl.umap(ad_skin, color=["cell_type", "study", "disease", "stage_class"], ncols=2)